# SQL — Technical Reference

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Core query structure | Any SQL query | §1 |
| Aggregations | Summarize rows | §2 |
| Conditional logic | Label or bucket values | §3 |
| Joins | Combine tables | §4 |
| Subqueries & CTEs | Multi-step logic | §5 |
| Window functions | Rank, lag/lead, running totals | §6 |
| Date & time | Date arithmetic, formatting | §7 |
| NULL handling | Missing values | §8 |
| Product metrics | DAU/WAU/MAU, cohort, retention | §9 |
| Performance tips | Query optimization | §10 |

---
## When to Use

| Signal | Pattern to reach for |
| :--- | :--- |
| Filter rows before grouping | `WHERE` |
| Filter after aggregation | `HAVING` |
| Reuse a subquery multiple times | CTE (`WITH`) |
| Rank within a group | `DENSE_RANK() OVER (PARTITION BY ...)` |
| Compare a row to the previous/next row | `LAG` / `LEAD` |
| Running total within a group | `SUM() OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` |
| Keep all rows from one table, match where possible | `LEFT JOIN` |
| Pivot rows into columns | `SUM(CASE WHEN ... THEN ... END)` |
| Find records with no match in another table | `LEFT JOIN ... WHERE B.id IS NULL` |
| Consecutive streaks / gaps | Row number subtraction trick |

---
## §1 — Core Query Structure

```sql
SELECT   column1, AGG(column2)      -- 5: what to return
FROM     table_name                 -- 1: source
JOIN     other ON table.id = other.id  -- 2: combine
WHERE    condition                  -- 3: filter rows BEFORE grouping
GROUP BY column1                    -- 4: group
HAVING   AGG(column2) > 100         -- 5: filter AFTER grouping
ORDER BY column1 DESC               -- 6: sort
LIMIT    10;                        -- 7: cap rows
```

**Execution order (not writing order):**
`FROM` → `JOIN` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`

**Common mistakes:**
- Using a `SELECT` alias in `WHERE` — alias doesn't exist yet at that stage; use the expression directly or wrap in a subquery
- Filtering aggregated results with `WHERE` instead of `HAVING`
- Selecting a non-aggregated column that is not in `GROUP BY` — causes error in strict SQL modes

---
## §2 — Aggregations

```sql
SELECT
  COUNT(*)                    AS total_rows,      -- counts NULLs
  COUNT(col)                  AS non_null_rows,   -- excludes NULLs
  COUNT(DISTINCT user_id)     AS unique_users,
  SUM(sales)                  AS total_sales,
  AVG(price)                  AS avg_price,       -- NULLs excluded from AVG
  MIN(score)                  AS min_score,
  MAX(score)                  AS max_score
FROM t;
```

**Conditional aggregation — pivot rows into columns:**
```sql
SELECT
  region,
  SUM(CASE WHEN platform = 'iOS'     THEN revenue ELSE 0 END) AS ios_rev,
  SUM(CASE WHEN platform = 'Android' THEN revenue ELSE 0 END) AS android_rev,
  AVG(CASE WHEN platform = 'iOS'     THEN revenue END)        AS ios_avg  -- NULLs auto-excluded
FROM sales
GROUP BY region;
```

**Common mistakes:**
- Using `AVG(col)` when NULLs should be treated as 0 — use `SUM(col) / COUNT(*)` instead
- `COUNT(*)` vs `COUNT(col)` — `COUNT(*)` includes NULLs, `COUNT(col)` does not
- Forgetting `ELSE 0` in `SUM(CASE WHEN ...)` — NULL propagates and the sum becomes NULL

---
## §3 — Conditional Logic

```sql
-- CASE WHEN (portable across all SQL dialects)
SELECT
  user_id,
  CASE
    WHEN spend > 100              THEN 'High'
    WHEN spend BETWEEN 50 AND 100 THEN 'Medium'
    ELSE 'Low'
  END AS spender_type
FROM transactions;

-- IF shortcut (MySQL only)
SELECT user_id,
       IF(spend > 100, 'High', 'Low') AS spender_type
FROM transactions;
```

**Common mistakes:**
- Conditions in `CASE WHEN` are evaluated top to bottom — order matters; put the most specific condition first
- Omitting `ELSE` — unmatched rows return NULL, not an empty string
- Using `IF()` in non-MySQL databases — use `CASE WHEN` for portability

---
## §4 — Joins

```sql
-- INNER JOIN — only matching rows
SELECT * FROM A INNER JOIN B ON A.id = B.id;

-- LEFT JOIN — all rows from A, matched rows from B (NULL if no match)
SELECT * FROM A LEFT JOIN B ON A.id = B.id;

-- Find rows in A with NO match in B
SELECT A.* FROM A
LEFT JOIN B ON A.id = B.id
WHERE B.id IS NULL;

-- FULL OUTER JOIN — all rows from both (MySQL: emulate with UNION)
SELECT * FROM A LEFT JOIN B ON A.id = B.id
UNION
SELECT * FROM A RIGHT JOIN B ON A.id = B.id;

-- CROSS JOIN — cartesian product (every row of A × every row of B)
SELECT * FROM A CROSS JOIN B;

-- Self join — join a table to itself
SELECT a.employee_id, a.name, b.name AS manager
FROM employees a
LEFT JOIN employees b ON a.manager_id = b.employee_id;

-- Filter inside JOIN ON vs WHERE
-- ON + AND: filters BEFORE join — keeps all left rows (use with LEFT JOIN)
SELECT * FROM A
LEFT JOIN B ON A.id = B.id AND B.status = 'active';

-- WHERE: filters AFTER join — turns LEFT JOIN into INNER JOIN
SELECT * FROM A
LEFT JOIN B ON A.id = B.id
WHERE B.status = 'active';           -- rows with no match are now excluded
```

**Common mistakes:**
- Filtering a `LEFT JOIN` result with `WHERE B.col = ...` — this silently converts it to an `INNER JOIN`; use `AND` in the `ON` clause instead
- Forgetting that `FULL OUTER JOIN` is not supported in MySQL — use `LEFT JOIN UNION RIGHT JOIN`
- Joining on columns with different data types — causes implicit cast, may skip index

---
## §5 — Subqueries & CTEs

```sql
-- Inline subquery
SELECT user_id, score
FROM (
  SELECT user_id, score, DENSE_RANK() OVER (ORDER BY score DESC) AS rnk
  FROM Scores
) ranked
WHERE rnk = 2;

-- CTE (WITH clause) — same result, more readable
WITH ranked AS (
  SELECT user_id, score, DENSE_RANK() OVER (ORDER BY score DESC) AS rnk
  FROM Scores
)
SELECT user_id, score FROM ranked WHERE rnk = 2;

-- Multiple CTEs
WITH
  cte1 AS (SELECT ...),
  cte2 AS (SELECT ... FROM cte1)
SELECT * FROM cte2;

-- Recursive CTE — hierarchical data (org chart, paths)
WITH RECURSIVE hierarchy AS (
  SELECT id, name, manager_id, 0 AS depth
  FROM employees
  WHERE manager_id IS NULL          -- anchor: root nodes
  UNION ALL
  SELECT e.id, e.name, e.manager_id, h.depth + 1
  FROM employees e
  JOIN hierarchy h ON e.manager_id = h.id  -- recursive step
)
SELECT * FROM hierarchy;
```

**CTE vs subquery — when to use each:**
- Use **CTE** when the subquery is referenced more than once, or when readability matters
- Use **subquery** for simple one-off filters — less overhead in some engines
- Use **recursive CTE** only when the data is truly hierarchical — avoid for flat problems

**Common mistakes:**
- Referencing a CTE alias before it is defined — CTEs execute in the order they are written
- Forgetting `UNION ALL` (not `UNION`) in recursive CTEs — `UNION` deduplicates and breaks most recursive logic
- Overusing CTEs for every intermediate step — unnecessary CTEs add overhead in some engines

---
## §6 — Window Functions

```sql
-- Syntax
FUNCTION() OVER (
  PARTITION BY col     -- reset window per group
  ORDER BY col         -- ordering within window
  ROWS BETWEEN ...     -- optional frame
)

-- Ranking
SELECT user_id, score,
  ROW_NUMBER()  OVER (ORDER BY score DESC) AS row_num,   -- unique, no ties
  RANK()        OVER (ORDER BY score DESC) AS rnk,       -- ties share rank, gaps after
  DENSE_RANK()  OVER (ORDER BY score DESC) AS dense_rnk  -- ties share rank, no gaps
FROM Scores;
```

**Rank functions comparison:**

| Function | Ties | Gaps after ties |
| :--- | :--- | :--- |
| `ROW_NUMBER()` | No — always unique | n/a |
| `RANK()` | Yes — share rank | Yes |
| `DENSE_RANK()` | Yes — share rank | No |

```sql
-- LAG / LEAD — access previous / next row
SELECT user_id, dt,
  LAG(value,  1) OVER (PARTITION BY user_id ORDER BY dt) AS prev_val,
  LEAD(value, 1) OVER (PARTITION BY user_id ORDER BY dt) AS next_val
FROM t;

-- Running total — ROWS BETWEEN
SELECT user_id, dt,
  SUM(amount) OVER (
    PARTITION BY user_id
    ORDER BY dt
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS run_sum
FROM payments;

-- Moving average (last 7 rows)
AVG(amount) OVER (
  PARTITION BY user_id
  ORDER BY dt
  ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
)

-- Kth row using LIMIT + OFFSET (when window functions unavailable)
SELECT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1;  -- 2nd highest
```

**Common mistakes:**
- Using `GROUP BY` alongside a window function on the same column — `GROUP BY` collapses rows; window functions operate before collapsing; if both are needed, use a CTE
- Omitting `ORDER BY` inside `OVER()` for `LAG`/`LEAD` — result is non-deterministic
- Confusing `ROWS BETWEEN` (physical rows) with `RANGE BETWEEN` (logical value range)

---
## §7 — Date & Time

```sql
-- Extract components
SELECT
  DATE(event_ts)                        AS date_only,
  EXTRACT(YEAR  FROM event_ts)          AS yr,
  EXTRACT(MONTH FROM event_ts)          AS mo,
  EXTRACT(DOW   FROM event_ts)          AS day_of_week,
  DATE_FORMAT(event_ts, '%Y-%m')        AS year_month   -- MySQL
FROM t;

-- Arithmetic
DATE_ADD(order_date, INTERVAL 7 DAY)    -- add 7 days
DATE_SUB(order_date, INTERVAL 1 MONTH) -- subtract 1 month
DATEDIFF(end_date, start_date)          -- days between (MySQL: end - start)

-- Current date/time
CURRENT_DATE                            -- date only
CURRENT_TIMESTAMP                       -- date + time
NOW()                                   -- MySQL alias for CURRENT_TIMESTAMP

-- Unix timestamp conversion
-- 10-digit = seconds, 13-digit = milliseconds
FROM_UNIXTIME(ts_col)                   -- seconds → datetime (MySQL)
DATE(FROM_UNIXTIME(ts_col))             -- seconds → date only (MySQL)
FROM_UNIXTIME(ts_col / 1000)            -- milliseconds → datetime (MySQL)
TO_TIMESTAMP(ts_col)                    -- seconds → timestamptz (PostgreSQL)
TO_TIMESTAMP(ts_col)::DATE              -- seconds → date only (PostgreSQL)
```

**Common mistakes:**
- `DATEDIFF` argument order differs by dialect — MySQL is `(end, start)`; PostgreSQL uses subtraction `end_date - start_date`
- Comparing a `DATETIME` column to a `DATE` string without casting — may miss rows or cause full table scan
- Using `DATE_FORMAT` (MySQL-only) in PostgreSQL — use `TO_CHAR(date, 'YYYY-MM')` instead

---
## §8 — NULL Handling

```sql
-- COALESCE — return first non-NULL (portable)
SELECT COALESCE(phone, email, 'Unknown') AS contact FROM people;

-- IFNULL — MySQL shorthand for two-argument COALESCE
SELECT IFNULL(salary, 0) AS salary FROM people;

-- IS NULL / IS NOT NULL — never use = NULL
SELECT * FROM t WHERE col IS NULL;
SELECT * FROM t WHERE col IS NOT NULL;

-- NULL in aggregates
COUNT(*)        -- includes NULLs
COUNT(col)      -- excludes NULLs
SUM(col)        -- ignores NULLs (returns NULL if all are NULL)
AVG(col)        -- ignores NULLs in both numerator and denominator
```

**Common mistakes:**
- Writing `WHERE col = NULL` — always evaluates to unknown (false); use `IS NULL`
- Assuming `AVG` treats NULLs as 0 — it ignores them; use `SUM / COUNT(*)` to include NULLs as 0
- Forgetting that `NULL` in a `JOIN` key never matches another `NULL` — two NULLs are not equal in SQL

---
## §9 — Product Metrics Patterns

```sql
-- DAU / WAU / MAU
SELECT
  COUNT(DISTINCT CASE WHEN event_date >= CURRENT_DATE - INTERVAL 1 DAY  THEN user_id END) AS DAU,
  COUNT(DISTINCT CASE WHEN event_date >= CURRENT_DATE - INTERVAL 7 DAY  THEN user_id END) AS WAU,
  COUNT(DISTINCT CASE WHEN event_date >= CURRENT_DATE - INTERVAL 30 DAY THEN user_id END) AS MAU
FROM events;

-- Cohort retention skeleton
WITH cohort AS (
  SELECT user_id, MIN(DATE(event_ts)) AS cohort_date
  FROM events
  GROUP BY user_id
),
activity AS (
  SELECT e.user_id,
         DATEDIFF(DATE(e.event_ts), c.cohort_date) / 7 AS week_num
  FROM events e
  JOIN cohort c ON e.user_id = c.user_id
)
SELECT cohort_date, week_num, COUNT(DISTINCT user_id) AS retained_users
FROM activity
JOIN cohort USING (user_id)
GROUP BY 1, 2
ORDER BY 1, 2;

-- Funnel analysis skeleton
SELECT
  COUNT(DISTINCT CASE WHEN step = 'view'    THEN user_id END) AS step1_view,
  COUNT(DISTINCT CASE WHEN step = 'cart'    THEN user_id END) AS step2_cart,
  COUNT(DISTINCT CASE WHEN step = 'checkout'THEN user_id END) AS step3_checkout
FROM events;

-- Gaps & Islands (consecutive streaks)
-- Trick: date - ROW_NUMBER() is constant within a streak
WITH numbered AS (
  SELECT user_id, login_date,
         login_date - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS grp
  FROM logins
)
SELECT user_id, MIN(login_date) AS streak_start,
       MAX(login_date) AS streak_end,
       COUNT(*) AS streak_length
FROM numbered
GROUP BY user_id, grp;
```

**Common mistakes:**
- Using `COUNT(DISTINCT user_id)` in a cohort query without proper date filtering — inflates retention numbers
- Gaps & Islands: using `RANK()` instead of `ROW_NUMBER()` — ties break the subtraction trick; always use `ROW_NUMBER()`
- Funnel: using `WHERE step = 'view'` instead of `CASE WHEN` — collapses to only view-step rows, losing downstream steps

---
## §10 — Performance Tips

```sql
-- Always filter early — reduce rows before joins and aggregations
SELECT * FROM orders
WHERE order_date >= '2024-01-01'    -- filter before joining
JOIN ...

-- Use EXPLAIN to inspect query plan
EXPLAIN SELECT ...
EXPLAIN ANALYZE SELECT ...          -- PostgreSQL: also runs the query

-- Avoid functions on indexed columns in WHERE — prevents index use
WHERE YEAR(created_at) = 2024       -- bad: full table scan
WHERE created_at >= '2024-01-01'
  AND created_at <  '2025-01-01'    -- good: index can be used
```

- Avoid `SELECT *` — select only needed columns to reduce I/O
- Index columns used in `JOIN ON`, `WHERE`, and `ORDER BY`
- Avoid correlated subqueries in `SELECT` — they run once per row; use a `JOIN` or `CTE` instead
- `COUNT(DISTINCT ...)` is expensive on large tables — consider approximate methods (`HLL`) in analytical engines
- Prefer `UNION ALL` over `UNION` unless deduplication is explicitly needed — `UNION` sorts and deduplicates

---
# Decision Guide

```
Filtering rows?
├── Before aggregation      → WHERE
└── After aggregation       → HAVING

Multi-step logic?
├── Used once               → Inline subquery
├── Used more than once     → CTE (WITH)
└── Hierarchical / recursive→ Recursive CTE (WITH RECURSIVE)

Combining tables?
├── Only matching rows      → INNER JOIN
├── All from left + matches → LEFT JOIN
├── Find no-match rows      → LEFT JOIN ... WHERE B.id IS NULL
├── All rows from both      → FULL OUTER JOIN (or LEFT UNION RIGHT in MySQL)
└── Every combination       → CROSS JOIN

Filter on joined table?
├── Want to keep all left rows   → Filter in ON clause with AND
└── Want only matched rows       → Filter in WHERE clause

Ranking?
├── Need unique rank per row     → ROW_NUMBER()
├── Ties share rank, gaps ok     → RANK()
└── Ties share rank, no gaps     → DENSE_RANK()

Aggregation across rows without collapsing?
└── Window function — OVER (PARTITION BY ... ORDER BY ...)
    ├── Running total            → SUM() OVER (ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
    ├── Previous row value       → LAG(col, 1)
    └── Next row value           → LEAD(col, 1)

NULL replacement?
├── Multiple fallbacks           → COALESCE(a, b, c)
└── Single fallback (MySQL)      → IFNULL(col, default)
```